In [12]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

here = Path.cwd()
ROOT = here if (here / "data" / "processed").exists() else here.parent
PROCESSED = ROOT / "data" / "processed"

# load the untouched master
master = gpd.read_file(PROCESSED / "yield_master.gpkg")

print("master loaded:", len(master), "rows,", len(master.columns), "columns")
print("flagged rows:", (master["qc_flag"] != "").sum())

master loaded: 1207 rows, 23 columns
flagged rows: 36


In [13]:
# work on a copy; master stays untouched
fixed = master.copy()

# ---- FIX 1: drop soil chemistry (80% placeholder -99/0, unusable) ----
chem_cols = ["carbon", "ph", "ec", "nitrogen", "phosphorus", "potassium"]
fixed = fixed.drop(columns=chem_cols)

print("dropped soil chemistry columns:", chem_cols)
print("columns now:", len(fixed.columns))
print(list(fixed.columns))

dropped soil chemistry columns: ['carbon', 'ph', 'ec', 'nitrogen', 'phosphorus', 'potassium']
columns now: 17
['uid', 'year', 'taluka', 'village', 'crop', 'variety', 'irrigation', 'crop_yield', 'sowing_date', 'harvest_date', 'area_acres', 'soil_color', 'soil_structure', 'soil_texture', 'soil_depth', 'qc_flag', 'geometry']


In [14]:
# ---- FIX 2: drop the 3 impossible yields (720, 260, 0) ----
# impossible = yield <= 0, or absurdly high (> 100). Real high yields (20-47) are NOT touched.
impossible = fixed[(fixed["crop_yield"] <= 0) | (fixed["crop_yield"] > 100)]

print("dropping these impossible-yield fields:")
print(impossible[["uid", "year", "village", "crop_yield"]].to_string(index=False))

before = len(fixed)
fixed = fixed[~((fixed["crop_yield"] <= 0) | (fixed["crop_yield"] > 100))].copy()
print()
print("rows before:", before, "-> after:", len(fixed), "(dropped", before - len(fixed), ")")

dropping these impossible-yield fields:
           uid  year village  crop_yield
3264-3546-2147  2023   Bhadi       260.0
4847-5235-3753  2023  Umarga       720.0
4324-4625-6825  2025 Jawalga         0.0

rows before: 1207 -> after: 1204 (dropped 3 )


In [15]:
# ---- FIX 3: correct year-typo sowing dates, blank the truly bad ones ----
# bad = sowing year does not match the field's crop year
bad = fixed["sowing_date"].notna() & (fixed["sowing_date"].dt.year != fixed["year"])

corrected, blanked = 0, 0
for idx in fixed.index[bad]:
    d = fixed.at[idx, "sowing_date"]
    target_year = fixed.at[idx, "year"]
    # kharif soybean sowing is normally June or July -> safe to fix the year only
    if d.month in (6, 7):
        fixed.at[idx, "sowing_date"] = d.replace(year=target_year)
        corrected += 1
    else:
        # month itself is suspicious (e.g. Aug/Sep) -> blank it
        fixed.at[idx, "sowing_date"] = pd.NaT
        blanked += 1

print("corrected (year typo, month kept):", corrected)
print("blanked (month also suspicious):", blanked)
print()

# verify: any sowing dates still with wrong year?
still_bad = fixed["sowing_date"].notna() & (fixed["sowing_date"].dt.year != fixed["year"])
print("remaining wrong-year sowing dates:", int(still_bad.sum()), "(should be 0)")
print("sowing_date missing now:", int(fixed["sowing_date"].isna().sum()))

corrected (year typo, month kept): 15
blanked (month also suspicious): 2

remaining wrong-year sowing dates: 0 (should be 0)
sowing_date missing now: 36


In [ ]:
# ---- VARIETY STANDARDIZATION: review the mapping (display only, no changes yet) ----
# The same variety appears under many spellings. This maps each spelling to one clean label.

variety_map = {
    # KDS 753
    "KDS 753": "KDS 753", "753": "KDS 753", "kds753": "KDS 753", "kds 753": "KDS 753",
    "KDS753": "KDS 753", "Kds 753": "KDS 753", "KDS-753": "KDS 753", "753 kds": "KDS 753",
    "Kds753": "KDS 753", "kda753": "KDS 753", "KSS753": "KDS 753",
    # DS 228
    "DS 228": "DS 228", "228": "DS 228", "DS228": "DS 228", "ds 228": "DS 228",
    "Ds228": "DS 228", "ds228": "DS 228", "Ds 228": "DS 228",
    # KDS 992
    "KDS 992": "KDS 992", "992": "KDS 992", "kds 992": "KDS 992",
    "केडीएस ९९२": "KDS 992", "992 किमया": "KDS 992",
    # MAUS 612
    "MAUS 612": "MAUS 612", "612": "MAUS 612", "maus 612": "MAUS 612",
    "MAUS -612": "MAUS 612", "Maus612": "MAUS 612", "एमएयुएस ६१२": "MAUS 612",
    # JS 335
    "JS 335": "JS 335", "335": "JS 335", "js 335": "JS 335", "js-335": "JS 335",
    "335 mahabij": "JS 335", "335(mahabij)": "JS 335", "335 oswal": "JS 335", "335oswal": "JS 335",
    # KDS 726
    "KDS 726": "KDS 726", "726": "KDS 726", "kds726": "KDS 726", "KDS-726": "KDS 726",
    "Kds-726": "KDS 726", "kds 726": "KDS 726", "726kds": "KDS 726", "726KDS": "KDS 726", "726 kds": "KDS 726",
    # KDS 725
    "KDS 725": "KDS 725",
    # MAUS 71  (KDS-71 and MAUS 158 left OUT here, see uncertain list below)
    "MAUS-71": "MAUS 71", "maus-71": "MAUS 71", "MAUS 71": "MAUS 71", "71": "MAUS 71", "71 mahabij": "MAUS 71",
    # AMS 1001
    "AMS 1001": "AMS 1001", "1001": "AMS 1001",
    # MAUS 162
    "MAUS 162": "MAUS 162", "162": "MAUS 162",
}

# show what each clean variety will collect, and its total count
raw = clean_only["variety"].astype(str).str.strip() if "clean_only" in dir() else fixed["variety"].astype(str).str.strip()
review = pd.DataFrame({"raw": raw})
review["clean"] = review["raw"].map(variety_map).fillna("UNMAPPED")

print("=== spellings grouped under each clean variety ===")
for clean_name, grp in review.groupby("clean"):
    spellings = grp["raw"].value_counts()
    total = spellings.sum()
    print(f"\n{clean_name}  (total {total} fields)")
    print("   ", dict(spellings))

print("\n\n=== STILL UNMAPPED (need decisions) ===")
unmapped = review[review["clean"] == "UNMAPPED"]["raw"].value_counts()
print(unmapped.to_string())

=== spellings grouped under each clean variety ===

AMS 1001  (total 12 fields)
    {'AMS 1001': np.int64(11), '1001': np.int64(1)}

DS 228  (total 292 fields)
    {'DS 228': np.int64(221), '228': np.int64(46), 'DS228': np.int64(18), 'ds 228': np.int64(3), 'Ds228': np.int64(2), 'ds228': np.int64(1), 'Ds 228': np.int64(1)}

JS 335  (total 99 fields)
    {'JS 335': np.int64(61), '335': np.int64(32), 'js 335': np.int64(2), '335oswal': np.int64(1), '335(mahabij)': np.int64(1), '335 mahabij': np.int64(1), 'js-335': np.int64(1)}

KDS 725  (total 18 fields)
    {'KDS 725': np.int64(18)}

KDS 726  (total 48 fields)
    {'KDS 726': np.int64(20), '726': np.int64(17), 'kds726': np.int64(4), 'KDS-726': np.int64(3), 'Kds-726': np.int64(1), '726kds': np.int64(1), '726KDS': np.int64(1), 'kds 726': np.int64(1)}

KDS 753  (total 463 fields)
    {'KDS 753': np.int64(250), '753': np.int64(89), 'kds753': np.int64(48), 'kds 753': np.int64(43), 'KDS753': np.int64(24), 'Kds 753': np.int64(3), 'Kds753': np.in

In [17]:
# ---- FIX 4: recompute area from the polygon (fixes negative/zero areas) ----
# project to UTM 43N (EPSG:32643) so area is measured correctly in meters, then to acres
fixed_utm = fixed.to_crs(32643)
area_m2 = fixed_utm.geometry.area
area_acres_new = area_m2 / 4046.86

# keep the old area for comparison
fixed["area_acres_old"] = fixed["area_acres"]
fixed["area_acres"] = area_acres_new.values

# report what changed
print("area recomputed from polygons.")
print("old area: min", round(fixed['area_acres_old'].min(), 2),
      "| negatives:", int((fixed['area_acres_old'] <= 0).sum()))
print("new area: min", round(fixed['area_acres'].min(), 2),
      "| negatives:", int((fixed['area_acres'] <= 0).sum()))
print()

# show the fields that had bad old area, now fixed
was_bad = fixed["area_acres_old"] <= 0
print("fields that had bad area (old -> new):")
print(fixed.loc[was_bad, ["uid", "village", "area_acres_old", "area_acres"]].round(3).to_string(index=False))

area recomputed from polygons.
old area: min -2.29 | negatives: 7
new area: min 0.26 | negatives: 0

fields that had bad area (old -> new):
           uid   village  area_acres_old  area_acres
7484-8598-6981   Warwada          -0.326       1.004
7494-8605-6990   Warwada          -0.022       1.098
7435-8541-6852   Dapkyal          -0.637       2.121
7398-8500-6801  Zari Kh.          -0.342       2.672
7423-8528-6832 Renapur 1          -0.062       2.726
7425-8530-6834   Harwadi          -0.004       1.737
7462-8569-6925   Harwadi          -2.292       2.859


In [18]:
# ---- drop the temporary comparison column ----
fixed = fixed.drop(columns=["area_acres_old"])

# ---- final summary before saving ----
print("FINAL analysis-ready table:")
print("rows:", len(fixed), "(master had 1207, dropped 3 impossible yields)")
print("columns:", len(fixed.columns))
print(list(fixed.columns))
print()
print("rows per year:")
print(fixed["year"].value_counts().sort_index())
print()
print("missing values remaining (kept as honest blanks, filled later in modeling):")
print(fixed.drop(columns="geometry").isna().sum()[lambda s: s > 0].to_string())
print()

# ---- save ----
out = PROCESSED / "yield_analysis_ready.gpkg"
fixed.to_file(out, driver="GPKG")
print("saved:", out)

FINAL analysis-ready table:
rows: 1204 (master had 1207, dropped 3 impossible yields)
columns: 17
['uid', 'year', 'taluka', 'village', 'crop', 'variety', 'irrigation', 'crop_yield', 'sowing_date', 'harvest_date', 'area_acres', 'soil_color', 'soil_structure', 'soil_texture', 'soil_depth', 'qc_flag', 'geometry']

rows per year:
year
2022     55
2023     58
2024    323
2025    768
Name: count, dtype: int64

missing values remaining (kept as honest blanks, filled later in modeling):
variety            34
sowing_date        36
area_acres          1
soil_color        226
soil_structure    226
soil_texture      226
soil_depth        226

saved: f:\THESIS\mtech_thesis_crop_yield_estimation\data\processed\yield_analysis_ready.gpkg
